### Importando a base de dados

In [ ]:
import pandas as pd
import requests
import urllib3
import csv
from io import BytesIO

In [ ]:

# ---------------------------------------------------------
# CONFIGURAÇÕES
# ---------------------------------------------------------

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

CSV_URLS = [
    "https://dados.ma.gov.br/sites/default/files/DESPESA_2025_01.csv",
    "https://dados.ma.gov.br/sites/default/files/DESPESA_2025_02.csv",
    "https://dados.ma.gov.br/sites/default/files/DESPESA_2025_03.csv",
    "https://dados.ma.gov.br/sites/default/files/DESPESA_2025_04.csv",
    "https://dados.ma.gov.br/sites/default/files/DESPESA_2025_05_1.csv",
    "https://dados.ma.gov.br/sites/default/files/DESPESA_2025_06.csv",
    "https://dados.ma.gov.br/sites/default/files/DESPESA_2025_07.csv",
    "https://dados.ma.gov.br/sites/default/files/DESPESA_2025_08.csv",
    "https://dados.ma.gov.br/sites/default/files/DESPESA_2025_09.csv",
    "https://dados.ma.gov.br/sites/default/files/DESPESA_2025_10.csv",
    "https://dados.ma.gov.br/sites/default/files/DESPESA_2025_11.csv",
    "https://dados.ma.gov.br/sites/default/files/DESPESA_2025_12.csv",
]

HEADERS = {
    "User-Agent": "Auditoria-Forense-Benford/1.0"
}

COLUNAS = [
    "ano",
    "mes",
    "fase",
    "codigo_unidade",
    "unidade",
    "valor",
    "codigo_credor",
    "credor_nome",
    "codigo_funcao",
    "codigo_natureza",
    "cod_grupo_despesa"
]

# ---------------------------------------------------------
# FUNÇÃO DE CARGA ROBUSTA (CHUNKS + SEM ASPAS)
# ---------------------------------------------------------

def carregar_despesas_chunks(chunksize=40_000):
    dataframes = []

    for url in CSV_URLS:
        print(f"Lendo: {url}")

        response = requests.get(
            url,
            headers=HEADERS,
            timeout=90,
            verify=False
        )
        response.raise_for_status()

        for chunk in pd.read_csv(
            BytesIO(response.content),
            sep=";",
            decimal=",",
            encoding="latin1",
            engine="python",
            quoting=csv.QUOTE_NONE,   # <-- CHAVE DA SOLUÇÃO
            escapechar="\\",          # <-- EVITA QUEBRA
            on_bad_lines="skip",
            usecols=lambda c: c in COLUNAS,
            chunksize=chunksize
        ):
            # Normalização imediata do valor
            chunk["valor"] = (
                chunk["valor"]
                .astype(str)
                .str.replace(".", "", regex=False)
                .str.replace(",", ".", regex=False)
            )

            chunk["valor"] = pd.to_numeric(chunk["valor"], errors="coerce")

            dataframes.append(chunk)

    df_final = pd.concat(dataframes, ignore_index=True)

    print("\nCarga concluída com sucesso")
    print("Total de registros:", len(df_final))

    return df_final

# ---------------------------------------------------------
# EXECUÇÃO
# ---------------------------------------------------------

if __name__ == "__main__":
    df = carregar_despesas_chunks()

    # Salva no diretório do projeto
    df.to_csv(
        "despesas_ma_2025_tratado.csv",
        sep=";",
        index=False,
        encoding="utf-8-sig"
    )

    print("Arquivo salvo com sucesso: despesas_ma_2025_tratado.csv")

    print("Tipos de dados:")
    print(df.dtypes)

    print("Amostra:")
    print(df.head())